# **Data Cleaning & Preprocessing**

**Purpose of this notebook / Mục đích của notebook**

This notebook turns the raw DataCo file into one clean dataset, then builds a star schema (dimension tables + one fact table) for PostgreSQL and Power BI. The steps are: (1) load the raw data, (2) inspect it, (3) drop columns we do not need, (4) clean it with `clean_data`, (5) save the clean file, (6) build the star schema, (7) check the keys, (8) save the tables.

*Tiếng Việt:* Notebook này biến file dữ liệu thô của DataCo thành một bộ dữ liệu sạch, sau đó xây star schema (các bảng dimension + một bảng fact) để dùng cho PostgreSQL và Power BI. Các bước: (1) đọc dữ liệu thô, (2) kiểm tra, (3) bỏ các cột không cần, (4) làm sạch bằng `clean_data`, (5) lưu file sạch, (6) xây star schema, (7) kiểm tra khóa, (8) lưu các bảng.

**Note / Lưu ý:** one row in this dataset is one **order item**, not one order (180,519 rows, but fewer orders). / Mỗi dòng trong dữ liệu là một **order item** (dòng hàng trong đơn), không phải một đơn hàng (180.519 dòng nhưng số đơn ít hơn).

In [5]:
!git clone https://github.com/jhsam007/supply-chain-analytics.git
%cd /content/supply-chain-analytics
import sys
sys.path.insert(0, "/content/supply-chain-analytics")
from src.preprocessing import load_data, clean_data, save_processed
print("Import successful")

Cloning into 'supply-chain-analytics'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 34 (delta 1), reused 5 (delta 0), pack-reused 27 (from 1)
Receiving objects: 100% (34/34), 50.17 MiB | 11.20 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Updating files: 100% (18/18), done.
/content/supply-chain-analytics
Import successful


In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load the raw data / Đọc dữ liệu thô**

The first cells import the libraries and clone the reference repository (it provides `clean_data`). Then we connect Google Drive and read the raw file `DataCoSupplyChainDataset.csv`. The file is not saved in UTF-8, so we read it with `encoding='latin1'` to avoid decoding errors. The result has 180,519 rows and 53 columns.

*Tiếng Việt:* Các cell đầu import thư viện và tải repo tham khảo (repo này cung cấp hàm `clean_data`). Sau đó mình kết nối Google Drive và đọc file dữ liệu thô `DataCoSupplyChainDataset.csv`. File không được lưu ở dạng UTF-8 nên mình đọc bằng `encoding='latin1'` để tránh lỗi giải mã. Kết quả có 180.519 dòng và 53 cột.

In [7]:
file_path = '/content/drive/MyDrive/BUS_450/DataCoSupplyChainDataset.csv'

In [8]:
df = pd.read_csv(file_path, encoding='latin1')

# **Dataset Overview**

In [9]:
df.shape

(180519, 53)

In [10]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [12]:
df.describe()

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Id,Customer Id,Customer Zipcode,Department Id,Latitude,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Price,Product Status
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180516.000000,180519.000000,180519.000000,...,180519.000000,180519.000000,180519.000000,180519.000000,24840.000000,180519.000000,180519.000000,0.0,180519.000000,180519.0
mean,3.497654,2.931847,21.974989,183.107609,0.548291,31.851451,6691.379495,35921.126914,5.443460,29.719955,...,2.127638,203.772096,183.107609,21.974989,55426.132327,692.509764,31.851451,NaN,141.232550,0.0
std,1.623722,1.374449,104.433526,120.043670,0.497664,15.640064,4162.918106,37542.461122,1.629246,9.813646,...,1.453451,132.273077,120.043670,104.433526,31919.279101,336.446807,15.640064,NaN,139.732492,0.0
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2.000000,1.000000,603.000000,2.000000,-33.937553,...,1.000000,9.990000,7.490000,-4274.979980,1040.000000,19.000000,2.000000,NaN,9.990000,0.0
25%,2.000000,2.000000,7.000000,104.379997,0.000000,18.000000,3258.500000,725.000000,4.000000,18.265432,...,1.000000,119.980003,104.379997,7.000000,23464.000000,403.000000,18.000000,NaN,50.000000,0.0
50%,3.000000,4.000000,31.520000,163.990005,1.000000,29.000000,6457.000000,19380.000000,5.000000,33.144863,...,1.000000,199.919998,163.990005,31.520000,59405.000000,627.000000,29.000000,NaN,59.990002,0.0
75%,5.000000,4.000000,64.800003,247.399994,1.000000,45.000000,9779.000000,78207.000000,7.000000,39.279617,...,3.000000,299.950012,247.399994,64.800003,90008.000000,1004.000000,45.000000,NaN,199.990005,0.0
max,6.000000,4.000000,911.799988,1939.989990,1.000000,76.000000,20757.000000,99205.000000,12.000000,48.781933,...,5.000000,1999.989990,1939.989990,911.799988,99301.000000,1363.000000,76.000000,NaN,1999.989990,0.0


In [13]:
df.isnull().sum()

,0
Type,0
Days for shipping (real),0
Days for shipment (scheduled),0
Benefit per order,0
Sales per customer,0
Delivery Status,0
Late_delivery_risk,0
Category Id,0
Category Name,0
Customer City,0


**Missing values and columns we drop / Giá trị thiếu và các cột bị bỏ**

The check above shows that most columns are complete. The exceptions are `Product Description` (completely empty), `Order Zipcode` (only 24,840 of 180,519 rows are filled), `Customer Lname` (8 missing) and `Customer Zipcode` (3 missing). In the next cell we drop 8 columns: the two that are empty or almost empty (`Product Description`, `Order Zipcode`), the personal or sensitive ones (`Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, `Customer Street`) and `Product Image`, which is only a link. None of them is needed for the analysis, and `Customer Id` is kept to identify customers.

*Tiếng Việt:* Kết quả kiểm tra ở trên cho thấy phần lớn các cột đều đầy đủ. Ngoại lệ là `Product Description` (trống hoàn toàn), `Order Zipcode` (chỉ có 24.840 trên 180.519 dòng có giá trị), `Customer Lname` (thiếu 8) và `Customer Zipcode` (thiếu 3). Ở cell tiếp theo mình bỏ 8 cột: hai cột trống hoặc gần như trống (`Product Description`, `Order Zipcode`), các cột thông tin cá nhân hoặc nhạy cảm (`Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, `Customer Street`) và `Product Image` vì chỉ là đường link. Không cột nào cần cho phân tích, và `Customer Id` vẫn được giữ để nhận diện khách hàng.

In [14]:
df = df.drop(columns=[
    "Product Description",
    "Order Zipcode",
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Image"
])

In [15]:
df.isnull().sum()

,0
Type,0
Days for shipping (real),0
Days for shipment (scheduled),0
Benefit per order,0
Sales per customer,0
Delivery Status,0
Late_delivery_risk,0
Category Id,0
Category Name,0
Customer City,0


In [16]:
df.dtypes

,0
Type,object
Days for shipping (real),int64
Days for shipment (scheduled),int64
Benefit per order,float64
Sales per customer,float64
Delivery Status,object
Late_delivery_risk,int64
Category Id,int64
Category Name,object
Customer City,object


**Clean the data with `clean_data` / Làm sạch dữ liệu bằng `clean_data`**

The table above still has the raw column names and data types. Here we apply `clean_data`, the cleaning function from the reference repository. It standardizes the column names (for example `Days for shipping (real)` becomes `days_for_shipping_real`), converts the date columns to real dates, and adds helper columns such as `shipping_delay` and `is_late`. Run `df.dtypes` again afterwards to compare with the table above.

*Tiếng Việt:* Bảng ở trên vẫn còn tên cột và kiểu dữ liệu gốc. Ở bước này mình áp dụng `clean_data`, hàm làm sạch lấy từ repo tham khảo. Hàm chuẩn hóa tên cột (ví dụ `Days for shipping (real)` thành `days_for_shipping_real`), chuyển các cột ngày sang kiểu ngày thật, và thêm các cột hỗ trợ như `shipping_delay` và `is_late`. Sau đó chạy lại `df.dtypes` để so sánh với bảng ở trên.

In [17]:
# Clean dataset
df = clean_data(df)

In [18]:
df.dtypes

,0
type,object
days_for_shipping_real,int64
days_for_shipment_scheduled,int64
benefit_per_order,float64
sales_per_customer,float64
delivery_status,object
late_delivery_risk,int64
category_id,int64
category_name,object
customer_city,object


**Save the cleaned data / Lưu dữ liệu đã làm sạch**

The cleaned table is saved to Google Drive as `clean_supply_chain.csv` (180,519 rows, 48 columns). The row count did not change, so cleaning only fixed and added columns and did not remove any order item. We use `utf-8-sig` so that Excel and Power BI read special characters correctly. This file is the input for the star schema below and for the forecasting and classification notebooks.

*Tiếng Việt:* Bảng đã làm sạch được lưu lên Google Drive với tên `clean_supply_chain.csv` (180.519 dòng, 48 cột). Số dòng không đổi, nghĩa là bước làm sạch chỉ sửa và thêm cột chứ không loại bỏ order item nào. Mình dùng `utf-8-sig` để Excel và Power BI đọc đúng ký tự đặc biệt. File này là đầu vào cho phần star schema bên dưới và cho các notebook forecasting và classification.

In [19]:
from google.colab import drive

drive.mount("/content/drive")

output_path = (
    "/content/drive/MyDrive/BUS_450/"
    "clean_supply_chain.csv"
)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to Google Drive:", output_path)
print("Data shape:", df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive: /content/drive/MyDrive/BUS_450/clean_supply_chain.csv
Data shape: (180519, 48)


# **star_schema**

**What is a star schema? / Star schema là gì?**

A star schema splits the clean table into small **dimension tables** (who, what, where, when) and one central **fact table** (the numbers). Here the fact table has one row per order item, and it points to six dimensions: product, customer, geography, shipping, order profile and date. The date dimension is used twice, once for the order date and once for the shipping date. Each dimension gets a surrogate key (`*_key`), a simple number that the fact table uses to join back. This layout makes SQL queries and Power BI reports faster and easier to build.

*Tiếng Việt:* Star schema tách bảng dữ liệu sạch thành các **bảng dimension** nhỏ (ai, cái gì, ở đâu, khi nào) và một **bảng fact** ở giữa (chứa các con số). Ở đây bảng fact có mỗi order item là một dòng, và nối tới sáu dimension: product, customer, geography, shipping, order profile và date. Dimension date được dùng hai lần: một lần cho ngày đặt hàng, một lần cho ngày giao hàng. Mỗi dimension có một surrogate key (`*_key`), là một con số đơn giản để bảng fact nối ngược lại. Cách tổ chức này giúp truy vấn SQL và báo cáo Power BI nhanh và dễ xây hơn.

 **Data preparation**

In [20]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/BUS_450/clean_supply_chain.csv", encoding="utf-8-sig")
print("Reloaded cleaned data:", df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reloaded cleaned data: (180519, 48)


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

work = df.copy()

work["order_date"] = pd.to_datetime(
    work["order_date"],
    errors="coerce"
)

work["shipping_date"] = pd.to_datetime(
    work["shipping_date"],
    errors="coerce"
)

print("Initial shape:", work.shape)

Initial shape: (180519, 48)


In [22]:
print("Rows:", len(work))
print(
    "Unique order-item IDs:",
    work["order_item_id"].nunique()
)
print(
    "Missing order-item IDs:",
    work["order_item_id"].isna().sum()
)

Rows: 180519
Unique order-item IDs: 180519
Missing order-item IDs: 0


**Dimension builder function**

In [23]:
def create_dimension(
    source_df,
    columns,
    key_name
):
    dimension = (
        source_df[columns]
        .drop_duplicates()
        .sort_values(
            by=columns,
            na_position="last"
        )
        .reset_index(drop=True)
    )

    dimension.insert(
        0,
        key_name,
        np.arange(1, len(dimension) + 1)
    )

    return dimension

**Create dim_product**

In [24]:
product_columns = [
    "product_card_id",
    "product_name",
    "product_status",
    "product_category_id",
    "category_name",
    "department_id",
    "department_name"
]

In [25]:
missing = [
    col for col in product_columns
    if col not in work.columns
]

print("Missing product columns:", missing)

Missing product columns: []


In [26]:
dim_product = create_dimension(
    source_df=work,
    columns=product_columns,
    key_name="product_key"
)

print("dim_product:", dim_product.shape)
display(dim_product.head())

dim_product: (118, 8)


,product_key,product_card_id,product_name,product_status,product_category_id,category_name,department_id,department_name
0,1,19,Nike Men's Fingertrap Max Training Shoe,0,2,Soccer,2,Fitness
1,2,24,Elevation Training Mask 2.0,0,2,Soccer,2,Fitness
2,3,35,adidas Brazuca 2014 Official Match Ball,0,3,Baseball & Softball,2,Fitness
3,4,37,adidas Kids' F5 Messi FG Soccer Cleat,0,3,Baseball & Softball,2,Fitness
4,5,44,adidas Men's F10 Messi TRX FG Soccer Cleat,0,3,Baseball & Softball,2,Fitness


**Create dim_customer**

In [27]:
customer_columns = [
    "customer_id",
    "customer_segment",
    "customer_city",
    "customer_state",
    "customer_country"
]

In [28]:
dim_customer = create_dimension(
    source_df=work,
    columns=customer_columns,
    key_name="customer_key"
)

print("dim_customer:", dim_customer.shape)
display(dim_customer.head())

dim_customer: (20652, 6)


,customer_key,customer_id,customer_segment,customer_city,customer_state,customer_country
0,1,1,Consumer,Brownsville,TX,EE. UU.
1,2,2,Consumer,Littleton,CO,EE. UU.
2,3,3,Consumer,Caguas,PR,Puerto Rico
3,4,4,Consumer,San Marcos,CA,EE. UU.
4,5,5,Home Office,Caguas,PR,Puerto Rico


**Create dim_geography**

In [29]:
geography_columns = [
    "market",
    "order_region",
    "order_country",
    "order_state",
    "order_city"
]

In [30]:
dim_geography = create_dimension(
    source_df=work,
    columns=geography_columns,
    key_name="geography_key"
)

print("dim_geography:", dim_geography.shape)
display(dim_geography.head())

dim_geography: (3772, 6)


,geography_key,market,order_region,order_country,order_state,order_city
0,1,Africa,Central Africa,Angola,Benguela,Benguela
1,2,Africa,Central Africa,Angola,Bié,Kuito
2,3,Africa,Central Africa,Angola,Huambo,Huambo
3,4,Africa,Central Africa,Angola,Huila,Lubango
4,5,Africa,Central Africa,Angola,Luanda,Luanda


**Create dim_shipping**

In [31]:
shipping_columns = [
    "shipping_mode"
]

dim_shipping = create_dimension(
    source_df=work,
    columns=shipping_columns,
    key_name="shipping_key"
)

print("dim_shipping:", dim_shipping.shape)
display(dim_shipping)

dim_shipping: (4, 2)


,shipping_key,shipping_mode
0,1,First Class
1,2,Same Day
2,3,Second Class
3,4,Standard Class


**Create dim_order_profile**

In [32]:
order_profile_columns = [
    "type",
    "order_status",
    "delivery_status"
]

In [33]:
dim_order_profile = create_dimension(
    source_df=work,
    columns=order_profile_columns,
    key_name="order_profile_key"
)

print(
    "dim_order_profile:",
    dim_order_profile.shape
)

display(dim_order_profile.head())

dim_order_profile: (23, 4)


,order_profile_key,type,order_status,delivery_status
0,1,CASH,CLOSED,Advance shipping
1,2,CASH,CLOSED,Late delivery
2,3,CASH,CLOSED,Shipping on time
3,4,DEBIT,COMPLETE,Advance shipping
4,5,DEBIT,COMPLETE,Late delivery


**Create dim_date**

In [34]:
minimum_date = min(
    work["order_date"].min(),
    work["shipping_date"].min()
)

maximum_date = max(
    work["order_date"].max(),
    work["shipping_date"].max()
)

print("Minimum date:", minimum_date)
print("Maximum date:", maximum_date)

Minimum date: 2015-01-01 00:00:00
Maximum date: 2018-02-06 22:14:00


In [35]:
date_range = pd.date_range(
    start=minimum_date.normalize(),
    end=maximum_date.normalize(),
    freq="D"
)

dim_date = pd.DataFrame({
    "full_date": date_range
})

In [36]:
dim_date["date_key"] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day_of_month"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1
dim_date["week_of_year"] = (
    dim_date["full_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

dim_date["year_month"] = (
    dim_date["full_date"]
    .dt.strftime("%Y-%m")
)

dim_date["is_weekend"] = (
    dim_date["full_date"].dt.dayofweek >= 5
)

In [37]:
dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "year_month",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend"
    ]
]

print("dim_date:", dim_date.shape)
display(dim_date.head())

dim_date: (1133, 12)


,date_key,full_date,year,quarter,month_number,month_name,year_month,week_of_year,day_of_month,day_of_week,day_name,is_weekend
0,20150101,2015-01-01,2015,1,1,January,2015-01,1,1,4,Thursday,False
1,20150102,2015-01-02,2015,1,1,January,2015-01,1,2,5,Friday,False
2,20150103,2015-01-03,2015,1,1,January,2015-01,1,3,6,Saturday,True
3,20150104,2015-01-04,2015,1,1,January,2015-01,1,4,7,Sunday,True
4,20150105,2015-01-05,2015,1,1,January,2015-01,2,5,1,Monday,False


**Attach foreign keys to the fact table**

In [38]:
fact = work.copy()

*product_key*

In [39]:
fact = fact.merge(
    dim_product,
    on=product_columns,
    how="left",
    validate="many_to_one"
)

*customer_key*

In [40]:
fact = fact.merge(
    dim_customer,
    on=customer_columns,
    how="left",
    validate="many_to_one"
)

*geography_key*

In [41]:
fact = fact.merge(
    dim_geography,
    on=geography_columns,
    how="left",
    validate="many_to_one"
)

*shipping_key*

In [42]:
fact = fact.merge(
    dim_shipping,
    on=shipping_columns,
    how="left",
    validate="many_to_one"
)

*order_profile_key*

In [43]:
fact = fact.merge(
    dim_order_profile,
    on=order_profile_columns,
    how="left",
    validate="many_to_one"
)

***Attach the two date keys***

In [44]:
date_key_mapping = (
    dim_date.set_index("full_date")["date_key"]
)

fact["order_date_key"] = (
    fact["order_date"]
    .dt.normalize()
    .map(date_key_mapping)
)

fact["shipping_date_key"] = (
    fact["shipping_date"]
    .dt.normalize()
    .map(date_key_mapping)
)

# **Select measures for the fact table**

**What goes into the fact table / Những gì đưa vào bảng fact**

The fact table keeps one row per order item. It holds (a) the keys that link to each dimension, (b) the sales and demand numbers (quantity, price, discount, revenue, profit) and (c) the delivery numbers (scheduled days, actual days, delay, late flags). Descriptive text such as product name or city is not repeated here, because it already lives in the dimension tables. The `rename` below only gives the columns clearer business names.

*Tiếng Việt:* Bảng fact giữ mỗi order item là một dòng. Bảng chứa (a) các khóa nối tới từng dimension, (b) các con số về bán hàng và nhu cầu (số lượng, giá, giảm giá, doanh thu, lợi nhuận) và (c) các con số về giao hàng (số ngày dự kiến, số ngày thực tế, độ trễ, cờ trễ). Các cột mô tả như tên sản phẩm hay thành phố không lặp lại ở đây vì đã nằm trong bảng dimension. Lệnh `rename` bên dưới chỉ đổi tên cột cho dễ hiểu hơn.

In [45]:
fact = fact.rename(
    columns={
        "order_item_quantity": "quantity",
        "order_item_product_price": "unit_price",
        "order_item_discount": "discount_amount",
        "order_item_discount_rate": "discount_rate",
        "sales": "gross_sales",
        "order_item_total": "net_revenue",
        "order_profit_per_order": "profit_amount",
        "order_item_profit_ratio": "profit_ratio",
        "days_for_shipment_scheduled":
            "scheduled_shipping_days",
        "days_for_shipping_real":
            "actual_shipping_days",
        "shipping_delay":
            "shipping_delay_days"
    }
)

In [46]:
fact.insert(
    0,
    "fact_key",
    np.arange(1, len(fact) + 1)
)

In [47]:
fact_columns = [
    "fact_key",

    # Business/degenerate keys
    "order_item_id",
    "order_id",

    # Foreign keys
    "order_date_key",
    "shipping_date_key",
    "product_key",
    "customer_key",
    "geography_key",
    "shipping_key",
    "order_profile_key",

    # Sales and demand measures
    "quantity",
    "unit_price",
    "discount_amount",
    "discount_rate",
    "gross_sales",
    "net_revenue",
    "profit_amount",
    "profit_ratio",

    # Logistics measures
    "scheduled_shipping_days",
    "actual_shipping_days",
    "shipping_delay_days",
    "late_delivery_risk",
    "is_late"
]

fact_order_items = fact[fact_columns].copy()

In [48]:
print("Fact table:", fact_order_items.shape)
display(fact_order_items.head())

Fact table: (180519, 23)


,fact_key,order_item_id,order_id,order_date_key,shipping_date_key,product_key,customer_key,geography_key,shipping_key,order_profile_key,...,discount_rate,gross_sales,net_revenue,profit_amount,profit_ratio,scheduled_shipping_days,actual_shipping_days,shipping_delay_days,late_delivery_risk,is_late
0,1,180517,77202,20180131,20180203,115,20650,2943,4,4,...,0.04,327.75,314.640015,91.250000,0.29,4,3,-1,0,False
1,2,179254,75939,20180113,20180118,115,19387,2790,4,18,...,0.05,327.75,311.359985,-249.089996,-0.80,4,5,1,1,True
2,3,179253,75938,20180113,20180117,115,19386,2790,4,3,...,0.06,327.75,309.720001,-247.779999,-0.80,4,4,0,0,False
3,4,179252,75937,20180113,20180116,115,19385,2621,4,4,...,0.07,327.75,304.809998,22.860001,0.08,4,3,-1,0,False
4,5,179251,75936,20180113,20180115,115,19384,2621,4,13,...,0.09,327.75,298.250000,134.210007,0.45,4,2,-2,0,False


**12. Check primary keys**

In [49]:
tables_and_keys = {
    "dim_product": (
        dim_product,
        "product_key"
    ),
    "dim_customer": (
        dim_customer,
        "customer_key"
    ),
    "dim_geography": (
        dim_geography,
        "geography_key"
    ),
    "dim_shipping": (
        dim_shipping,
        "shipping_key"
    ),
    "dim_order_profile": (
        dim_order_profile,
        "order_profile_key"
    ),
    "dim_date": (
        dim_date,
        "date_key"
    ),
    "fact_order_items": (
        fact_order_items,
        "fact_key"
    )
}

for table_name, (table_df, key) in tables_and_keys.items():
    print(
        table_name,
        "| rows:", len(table_df),
        "| unique keys:", table_df[key].nunique(),
        "| missing keys:", table_df[key].isna().sum()
    )

    assert table_df[key].notna().all()
    assert table_df[key].is_unique

dim_product | rows: 118 | unique keys: 118 | missing keys: 0
dim_customer | rows: 20652 | unique keys: 20652 | missing keys: 0
dim_geography | rows: 3772 | unique keys: 3772 | missing keys: 0
dim_shipping | rows: 4 | unique keys: 4 | missing keys: 0
dim_order_profile | rows: 23 | unique keys: 23 | missing keys: 0
dim_date | rows: 1133 | unique keys: 1133 | missing keys: 0
fact_order_items | rows: 180519 | unique keys: 180519 | missing keys: 0


**13. Check foreign keys**

In [50]:
foreign_keys = [
    "order_date_key",
    "shipping_date_key",
    "product_key",
    "customer_key",
    "geography_key",
    "shipping_key",
    "order_profile_key"
]

print(
    fact_order_items[foreign_keys]
    .isna()
    .sum()
)

order_date_key       0
shipping_date_key    0
product_key          0
customer_key         0
geography_key        0
shipping_key         0
order_profile_key    0
dtype: int64


In [51]:
assert fact_order_items["product_key"].isin(
    dim_product["product_key"]
).all()

assert fact_order_items["customer_key"].isin(
    dim_customer["customer_key"]
).all()

assert fact_order_items["geography_key"].isin(
    dim_geography["geography_key"]
).all()

assert fact_order_items["shipping_key"].isin(
    dim_shipping["shipping_key"]
).all()

assert fact_order_items["order_profile_key"].isin(
    dim_order_profile["order_profile_key"]
).all()

assert fact_order_items["order_date_key"].isin(
    dim_date["date_key"]
).all()

assert fact_order_items["shipping_date_key"].isin(
    dim_date["date_key"]
).all()

print("All relationships are valid.")

All relationships are valid.


In [52]:
assert len(fact_order_items) == len(work)

print(
    "Cleaned rows:",
    len(work)
)

print(
    "Fact rows:",
    len(fact_order_items)
)

Cleaned rows: 180519
Fact rows: 180519


**14. Save tables as CSV**

In [53]:
# output_folder = Path(
    "/content/drive/MyDrive/BUS_450/star_schema"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

IndentationError: unexpected indent (4123124.py, line 2)

In [ ]:
tables = {
    "dim_date.csv": dim_date,
    "dim_product.csv": dim_product,
    "dim_customer.csv": dim_customer,
    "dim_geography.csv": dim_geography,
    "dim_shipping.csv": dim_shipping,
    "dim_order_profile.csv": dim_order_profile,
    "fact_order_items.csv": fact_order_items
}

for filename, table_df in tables.items():
    output_path = output_folder / filename

    table_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        filename,
        "→",
        table_df.shape
    )

## Notes and checks / Ghi chú và kiểm tra

**What we checked / Những gì đã kiểm tra**

- The fact table has 180,519 rows, the same as the cleaned data, so no row was lost or duplicated when the dimensions were joined.
- Every primary key is unique, and no foreign key in the fact table is empty.
- Dimension sizes: `dim_product` 118, `dim_customer` 20,652, `dim_geography` 3,772, `dim_shipping` 4, `dim_order_profile` 23.

*Tiếng Việt:*

- Bảng fact có 180.519 dòng, bằng đúng số dòng của dữ liệu đã làm sạch, nên không có dòng nào bị mất hoặc nhân đôi khi ghép các dimension.
- Mọi khóa chính đều duy nhất, và không có khóa ngoại nào bị trống trong bảng fact.
- Kích thước các dimension: `dim_product` 118, `dim_customer` 20.652, `dim_geography` 3.772, `dim_shipping` 4, `dim_order_profile` 23.

**Things to remember when writing the thesis / Cần nhớ khi viết luận văn**

- **Two ways to say "late":** `late_delivery_risk` comes with the dataset, while `is_late` / `shipping_delay_days` is calculated from actual vs. scheduled days. The number of late items can differ between them, so choose one definition and state it clearly. / **Hai cách xác định "trễ":** `late_delivery_risk` có sẵn trong dữ liệu, còn `is_late` / `shipping_delay_days` được tính từ số ngày thực tế so với dự kiến. Số lượng đơn trễ có thể khác nhau giữa hai cách, nên cần chọn một định nghĩa và nêu rõ trong bài.
- **Time range:** orders run from 2015 to early 2018, but the later weeks look incomplete (very few products per week), so the forecasting notebook only uses weeks up to 2017-10-01. / **Khoảng thời gian:** đơn hàng từ 2015 đến đầu 2018, nhưng các tuần cuối có vẻ chưa đầy đủ (số sản phẩm mỗi tuần rất thấp), nên notebook forecasting chỉ dùng các tuần đến 2017-10-01.
- **File encoding:** the CSV files are saved with `utf-8-sig` (UTF-8 with a BOM). When reading them in pandas, use the same encoding; when loading into PostgreSQL, check that the BOM does not end up in the first column name. / **Mã hóa file:** các file CSV được lưu bằng `utf-8-sig` (UTF-8 có BOM). Khi đọc bằng pandas hãy dùng cùng encoding; khi nạp vào PostgreSQL cần kiểm tra BOM không dính vào tên cột đầu tiên.